## Step Functions — workflow orchestration

Chaining Lambda A → B → C works on day one and becomes a swamp by month three: each function has its own retry logic, error handling is inconsistent, there's no view of which step failed, and long flows hit Lambda's 15-minute ceiling. **Step Functions** is the managed answer — you describe the workflow as a **state machine** in **Amazon States Language (ASL)** (a JSON of states + transitions); Step Functions runs it, invokes each step, retries on failure, and shows the **full execution history** in the console.

**State types**: **Task** (invoke a Lambda or call an AWS service directly), **Choice** (if/else branch on input), **Wait** (pause for a duration or timestamp), **Parallel** (run branches at once, join), **Map** (iterate an array with a sub-workflow), plus Pass/Succeed/Fail. The error story is why it exists: **Retry** attaches rules per Task (error types, interval, max attempts, backoff — e.g. 2 s / 3 attempts / 2× absorbs transient Lambda throttles and DynamoDB throughput errors), and **Catch** routes to a handler state when retries are exhausted — turning a try/catch tangle into declarative workflow config.

Two workflow types: **Standard** — up to **1 year**, **exactly-once** per state, priced **per state transition**, history kept forever — for business workflows (fulfilment, approvals, pipelines). **Express** — up to **5 minutes**, at-least/at-most-once, priced per duration, **100k+ starts/sec** — for high-volume short flows (streaming/IoT event processing). And `.waitForTaskToken` **pauses for a human approval or external callback** for as long as needed.